# Bayesian Threshold (Rational Stick-or-Switch) Model

Implements the **Bayesian Threshold** model:
- Each stage, compute expected value of current role vs alternatives (given posterior beliefs)
- Only consider switching if the value gap exceeds a **threshold** (delta)
- If threshold met: **softmax** among only the roles that cleared the threshold
- If not met: **stick** with current role (deterministic)
- Stage 1: softmax over all roles (no current role to stick to)

**Parameters**: `tau_prior`, `tau_softmax`, `epsilon` (action noise), `delta` (switch threshold)

In [1]:
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from itertools import product
from scipy.optimize import minimize

# Shared package
from shared import EXPORTS_DIR, DATA_ROOT
from shared.constants import (
    F, T, M,
    ROLE_NAMES, ROLE_SHORT, ROLE_CHAR_TO_IDX, GAME_ROLE_TO_IDX,
    ALL_ROLE_COMBOS, EPSILON, MAX_STAGES, TURNS_PER_STAGE,
)
from shared.parsing import canonical_combo, get_canonical_combos
from shared.inference import (
    utility_based_prior, uniform_prior,
    bayesian_update, action_prob, preferred_action, game_step,
    softmax_role_dist, combo_marginal,
    ATTACK, DEFEND, HEAL,
)
from shared.evaluation import (
    run_predictions, compute_pearson, compute_log_likelihood, extract_metrics,
)

# Still need online_model_sim for load_team_rounds
OMS_DIR = Path(DATA_ROOT).parent.parent / 'computational_model' / 'analysis'
sys.path.insert(0, str(OMS_DIR))
import online_model_sim as oms

## 1. Load Data

In [2]:
# Monkey-patch env paths to use shared data directory
oms.VALUE_MATRICES_DIR = DATA_ROOT / 'human_envs_value_matrices'
oms.ENVS_DIR = DATA_ROOT / 'envs'

# Monkey-patch config loader to avoid JAX dependency
from shared.env_loading import load_env_config
import re as _re

def _load_config_no_jax(config_path):
    text = Path(config_path).read_text()
    team_max_hp = int(_re.search(r'TEAM_MAX_HP\s*=\s*(\d+)', text).group(1))
    enemy_max_hp = int(_re.search(r'ENEMY_MAX_HP\s*=\s*(\d+)', text).group(1))
    boss_damage = float(_re.search(r'BOSS_DAMAGE\s*=\s*([\d.]+)', text).group(1))
    ps_match = _re.search(r'PLAYER_STATS\s*=\s*(?:jnp\.array|np\.array)?\(?\s*(\[\[.+?\]\])\s*\)?', text, _re.DOTALL)
    rows = _re.findall(r'\[([^\[\]]+)\]', ps_match.group(1))
    player_stats = np.array([[float(x) for x in row.split(',')] for row in rows])
    class Config: pass
    cfg = Config()
    cfg.TEAM_MAX_HP = team_max_hp
    cfg.ENEMY_MAX_HP = enemy_max_hp
    cfg.BOSS_DAMAGE = boss_damage
    cfg.PLAYER_STATS = player_stats
    return cfg

oms.load_config_module = _load_config_no_jax

# Load data
DATA_DIRS = [
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-06-09-54-19'),
    str(EXPORTS_DIR / 'bayesian-role-specialization-2026-03-18-15-47-09'),
]

records = oms.load_team_rounds(data_dirs=DATA_DIRS)
n_envs = len(set(r['env_id'] for r in records))
print(f'Loaded {len(records)} team-rounds across {n_envs} environments')

Loaded 66 team-rounds across 6 environments


## 2. Bayesian Threshold Model

At each stage (s > 1):
1. Compute expected value of each role for each agent (given posterior beliefs)
2. For each agent, compute `delta(r) = V(r) - V(current_role)` for each alternative role
3. Collect candidate roles where `delta(r) > threshold`
4. If no candidates: stick with current role (deterministic)
5. If candidates exist: softmax over only those candidates

Stage 1: softmax over all roles (same as Bayesian-Value).

In [3]:
def expected_values_per_role(agent_i, intent, team_hp, enemy_hp, prior, values):
    """Compute expected value for each role of agent_i, marginalizing over others.

    Returns length-3 array of expected values (one per role).
    Same computation as softmax_role_dist but returns raw values before softmax.
    """
    other_agents = [a for a in range(3) if a != agent_i]
    other_probs = np.sum(prior, axis=agent_i)
    total = other_probs.sum()
    other_probs = other_probs / total if total > 0 else np.ones((3, 3)) / 9.0

    ev = np.zeros(3)
    for r_i in range(3):
        for r_j in range(3):
            for r_k in range(3):
                roles = [0, 0, 0]
                roles[agent_i] = r_i
                roles[other_agents[0]] = r_j
                roles[other_agents[1]] = r_k
                flat_idx = roles[0] * 9 + roles[1] * 3 + roles[2]
                ev[r_i] += other_probs[r_j, r_k] * float(values[flat_idx, intent, team_hp, enemy_hp])
    return ev


def threshold_role_dist(agent_i, intent, team_hp, enemy_hp, prior, values,
                        current_role, delta, tau):
    """Bayesian Threshold: stick unless value gap > delta, then softmax among candidates.

    Args:
        agent_i: Agent index (0, 1, or 2).
        intent: Enemy attack intent (0 or 1).
        team_hp, enemy_hp: Current HP values (int).
        prior: (3, 3, 3) joint posterior.
        values: (27, 2, H, W) value table.
        current_role: Index of the role played last stage.
        delta: Minimum value improvement to consider switching.
        tau: Softmax temperature over candidate roles.

    Returns:
        Length-3 probability array over roles.
    """
    ev = expected_values_per_role(agent_i, intent, team_hp, enemy_hp, prior, values)
    current_val = ev[current_role]

    # Find roles with sufficient improvement
    candidates = [r for r in range(3) if r != current_role and (ev[r] - current_val) > delta]

    if not candidates:
        # Stick with current role
        dist = np.zeros(3)
        dist[current_role] = 1.0
        return dist

    # Softmax over candidates only
    candidate_vals = np.array([ev[r] for r in candidates])
    scaled = candidate_vals / tau
    scaled -= scaled.max()
    exp_vals = np.exp(scaled)
    probs = exp_vals / exp_vals.sum()

    dist = np.zeros(3)
    for i, r in enumerate(candidates):
        dist[r] = probs[i]
    return dist

In [4]:
def make_bayesian_thresh(tau_prior=5.6, tau_softmax=0.1, epsilon=0.2, delta=0.5):
    """Bayesian Threshold: stick unless value gap > delta, then softmax among candidates."""
    def predict_fn(record):
        env = record['env_config']
        values = env['values']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        results = []
        turn_idx = 0
        prev_roles = None

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            intent = record['lds'][turn_idx]
            thp = int(min(max(0, team_hp), team_max_hp))
            ehp = int(min(max(0, enemy_hp), enemy_max_hp))

            # Compute per-agent role distribution
            per_agent = []
            for i in range(3):
                if prev_roles is None:
                    # Stage 1: softmax over all roles (same as Bayesian-Value)
                    per_agent.append(softmax_role_dist(i, intent, thp, ehp, prior, values, tau_softmax))
                else:
                    # Stage 2+: threshold-based stick-or-switch
                    per_agent.append(threshold_role_dist(
                        i, intent, thp, ehp, prior, values,
                        current_role=prev_roles[i], delta=delta, tau=tau_softmax))

            # Build joint predicted distribution
            predicted_dist = {}
            for r0 in range(3):
                for r1 in range(3):
                    for r2 in range(3):
                        combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                        predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

            results.append({
                'predicted_dist': predicted_dist,
                'human_combo': human_combo,
                'model_marginal': np.mean(per_agent, axis=0),
            })

            # Teacher-force: advance game with actual human roles
            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = human_roles
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        return results
    return predict_fn

## 4. Fine-Tuning: Grid Search

3-stage process:
1. Coarse grid search over all 4 parameters (tau_prior, tau_softmax, epsilon, delta)
2. Refined grid around the best point
3. Scipy polishing

Caching strategy: precompute trajectories (posterior + game state) per `(tau_prior, epsilon)`,
then sweep `(tau_softmax, delta)` cheaply from cache.

In [5]:
metric = 'combo_r'  # optimize combo Pearson r


def precompute_trajectories(records, tau_prior, epsilon):
    """Precompute posterior + game state per stage for each record.

    Depends only on (tau_prior, epsilon) and observed human actions.
    Returns list of trajectories (one per record), each a list of stage dicts.
    """
    trajectories = []
    for record in records:
        env = record['env_config']
        player_stats = env['player_stats']
        boss_damage = env['boss_damage']
        team_max_hp, enemy_max_hp = env['team_max_hp'], env['enemy_max_hp']

        team_hp = float(team_max_hp)
        enemy_hp = float(enemy_max_hp)
        prior = utility_based_prior(player_stats, tau=tau_prior)
        turn_idx = 0
        prev_roles = None
        stages = []

        for human_combo in record['stage_roles']:
            if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                break

            intent = record['lds'][turn_idx]
            thp = int(min(max(0, team_hp), team_max_hp))
            ehp = int(min(max(0, enemy_hp), enemy_max_hp))

            stages.append({
                'prior': prior.copy(),
                'intent': intent,
                'thp': thp,
                'ehp': ehp,
                'human_combo': human_combo,
                'prev_roles': prev_roles,
            })

            human_roles = [ROLE_CHAR_TO_IDX[c] for c in human_combo]
            prev_roles = list(human_roles)
            for _ in range(TURNS_PER_STAGE):
                if turn_idx >= len(record['lds']) or team_hp <= 0 or enemy_hp <= 0:
                    break
                intent_t = record['lds'][turn_idx]
                actions = [preferred_action(human_roles[i], intent_t, team_hp, team_max_hp)
                           for i in range(3)]
                prior = bayesian_update(prior, actions, intent_t, team_hp, team_max_hp, epsilon)
                team_hp, enemy_hp = game_step(intent_t, team_hp, enemy_hp, actions,
                                              player_stats, boss_damage, team_max_hp)
                turn_idx += 1

        trajectories.append(stages)
    return trajectories


def predict_from_trajectory_thresh(record, trajectory, values, tau_softmax, delta):
    """Generate predictions from precomputed trajectory using threshold model."""
    results = []
    for stage in trajectory:
        prior = stage['prior']
        intent, thp, ehp = stage['intent'], stage['thp'], stage['ehp']
        prev_roles = stage['prev_roles']
        human_combo = stage['human_combo']

        per_agent = []
        for i in range(3):
            if prev_roles is None:
                per_agent.append(softmax_role_dist(i, intent, thp, ehp, prior, values, tau_softmax))
            else:
                per_agent.append(threshold_role_dist(
                    i, intent, thp, ehp, prior, values,
                    current_role=prev_roles[i], delta=delta, tau=tau_softmax))

        predicted_dist = {}
        for r0 in range(3):
            for r1 in range(3):
                for r2 in range(3):
                    combo = ROLE_SHORT[r0] + ROLE_SHORT[r1] + ROLE_SHORT[r2]
                    predicted_dist[combo] = float(per_agent[0][r0] * per_agent[1][r1] * per_agent[2][r2])

        results.append({
            'predicted_dist': predicted_dist,
            'human_combo': human_combo,
            'model_marginal': np.mean(per_agent, axis=0),
        })
    return results


def evaluate_thresh(records, tau_prior, tau_softmax, epsilon, delta):
    """Run Bayesian Threshold at given params (uncached, for scipy)."""
    results = run_predictions(records, make_bayesian_thresh(
        tau_prior=tau_prior, tau_softmax=tau_softmax,
        epsilon=epsilon, delta=delta))
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'tau_softmax': tau_softmax,
        'epsilon': epsilon, 'delta': delta,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_from_cache_thresh(records, trajectories, tau_softmax, delta):
    """Evaluate using precomputed trajectories (fast)."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory_thresh(record, trajectories[idx],
                                              record['env_config']['values'],
                                              tau_softmax, delta)
    results = run_predictions(records, predict_fn)
    corrs = compute_pearson(results)
    ll = compute_log_likelihood(results)
    g = corrs.get('__global__', {})
    return {
        'tau_softmax': tau_softmax, 'delta': delta,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_thresh(records, tau_prior_vals, tau_softmax_vals, eps_vals, delta_vals):
    """Cached grid search: precompute per (tau_prior, eps), sweep (tau_softmax, delta)."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    inner_combos = list(product(tau_softmax_vals, delta_vals))
    total = len(outer_combos) * len(inner_combos)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for ts, d in inner_combos:
            count += 1
            if count % 100 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_from_cache_thresh(records, trajectories, ts, d)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results


def pick_best(results, metric='combo_r'):
    valid = [r for r in results if not np.isnan(r.get(metric, float('nan')))]
    if not valid:
        return results[0]
    return max(valid, key=lambda r: r[metric])

### Step 1: Coarse Grid

In [6]:
# Grid ranges
tau_prior_min,   tau_prior_max,   tau_prior_steps   = 0.1, 20.0, 10
tau_softmax_min, tau_softmax_max, tau_softmax_steps = 0.1, 20.0, 10
eps_min, eps_max, eps_steps = 0.0, 1.0, 21
# delta: threshold for switching — search from near-zero (always switch) to large (never switch)
delta_min, delta_max, delta_steps = 0.0, 100.0, 11
delta_vals = np.linspace(delta_min, delta_max, delta_steps)

tau_prior_vals   = np.linspace(tau_prior_min, tau_prior_max, tau_prior_steps)
tau_softmax_vals = np.linspace(tau_softmax_min, tau_softmax_max, tau_softmax_steps)
eps_vals = np.linspace(eps_min, eps_max, eps_steps)

total = len(tau_prior_vals) * len(tau_softmax_vals) * len(eps_vals) * len(delta_vals)
print(f'Coarse grid: {len(tau_prior_vals)} x {len(tau_softmax_vals)} x {len(eps_vals)} x {len(delta_vals)} = {total} points')
print(f'  tau_prior:    linspace({tau_prior_min}, {tau_prior_max}, {tau_prior_steps})')
print(f'  tau_softmax:  linspace({tau_softmax_min}, {tau_softmax_max}, {tau_softmax_steps})')
print(f'  epsilon:      linspace({eps_min}, {eps_max}, {eps_steps})')
print(f'  delta:        linspace({delta_min}, {delta_max}, {delta_steps})')
print()

sweep_results = grid_search_thresh(records, tau_prior_vals, tau_softmax_vals, eps_vals, delta_vals)
best = pick_best(sweep_results, metric)
print(f'\nCoarse best: tau_prior={best["tau_prior"]:.4f} tau_softmax={best["tau_softmax"]:.4f} '
      f'eps={best["epsilon"]:.6f} delta={best["delta"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Coarse grid: 10 x 10 x 21 x 11 = 23100 points
  tau_prior:    linspace(0.1, 20.0, 10)
  tau_softmax:  linspace(0.1, 20.0, 10)
  epsilon:      linspace(0.0, 1.0, 21)
  delta:        linspace(0.0, 100.0, 11)

  [100/23100] ...
  [200/23100] ...
  [300/23100] ...
  [400/23100] ...
  [500/23100] ...
  [600/23100] ...
  [700/23100] ...
  [800/23100] ...
  [900/23100] ...
  [1000/23100] ...
  [1100/23100] ...
  [1200/23100] ...
  [1300/23100] ...
  [1400/23100] ...
  [1500/23100] ...
  [1600/23100] ...
  [1700/23100] ...
  [1800/23100] ...
  [1900/23100] ...
  [2000/23100] ...
  [2100/23100] ...
  [2200/23100] ...
  [2300/23100] ...
  [2400/23100] ...
  [2500/23100] ...
  [2600/23100] ...
  [2700/23100] ...
  [2800/23100] ...
  [2900/23100] ...
  [3000/23100] ...
  [3100/23100] ...
  [3200/23100] ...
  [3300/23100] ...
  [3400/23100] ...
  [3500/23100] ...
  [3600/23100] ...
  [3700/23100] ...
  [3800/23100] ...
  [3900/23100] ...
  [4000/23100] ...
  [4100/23100] ...
  [4200/23100] ...
  [4

### Step 2: Refined Grid

In [7]:
# Refine around coarse best
tp_step = (tau_prior_max - tau_prior_min) / tau_prior_steps
ts_step = (tau_softmax_max - tau_softmax_min) / tau_softmax_steps
eps_step_size = (eps_max - eps_min) / (eps_steps - 1)
delta_step = (delta_max - delta_min) / delta_steps

fine_tp = np.linspace(
    max(tau_prior_min, best['tau_prior'] - tp_step),
    min(tau_prior_max, best['tau_prior'] + tp_step), 10)
fine_ts = np.linspace(
    max(tau_softmax_min, best['tau_softmax'] - ts_step),
    min(tau_softmax_max, best['tau_softmax'] + ts_step), 10)
fine_eps = np.linspace(
    max(eps_min, best['epsilon'] - eps_step_size),
    min(eps_max, best['epsilon'] + eps_step_size), 10)
fine_delta = np.linspace(
    max(delta_min, best['delta'] - delta_step),
    min(delta_max, best['delta'] + delta_step), 10)

total_fine = len(fine_tp) * len(fine_ts) * len(fine_eps) * len(fine_delta)
print(f'Refined grid: {total_fine} points')
print(f'  tau_prior:   [{fine_tp[0]:.2f}, {fine_tp[-1]:.2f}]')
print(f'  tau_softmax: [{fine_ts[0]:.2f}, {fine_ts[-1]:.2f}]')
print(f'  epsilon:     [{fine_eps[0]:.6f}, {fine_eps[-1]:.6f}]')
print(f'  delta:       [{fine_delta[0]:.2f}, {fine_delta[-1]:.2f}]')
print()

sweep_results2 = grid_search_thresh(records, fine_tp, fine_ts, fine_eps, fine_delta)
best = pick_best(sweep_results2, metric)
print(f'\nRefined best: tau_prior={best["tau_prior"]:.4f} tau_softmax={best["tau_softmax"]:.4f} '
      f'eps={best["epsilon"]:.6f} delta={best["delta"]:.4f}')
print(f'  combo_r={best["combo_r"]:.4f}  marg_r={best["marg_r"]:.4f}  mean_ll={best["mean_ll"]:.4f}')

Refined grid: 10000 points
  tau_prior:   [4.74, 8.72]
  tau_softmax: [2.53, 6.51]
  epsilon:     [0.900000, 1.000000]
  delta:       [30.91, 49.09]

  [100/10000] ...
  [200/10000] ...
  [300/10000] ...
  [400/10000] ...
  [500/10000] ...
  [600/10000] ...
  [700/10000] ...
  [800/10000] ...
  [900/10000] ...
  [1000/10000] ...
  [1100/10000] ...
  [1200/10000] ...
  [1300/10000] ...
  [1400/10000] ...
  [1500/10000] ...
  [1600/10000] ...
  [1700/10000] ...
  [1800/10000] ...
  [1900/10000] ...
  [2000/10000] ...
  [2100/10000] ...
  [2200/10000] ...
  [2300/10000] ...
  [2400/10000] ...
  [2500/10000] ...
  [2600/10000] ...
  [2700/10000] ...
  [2800/10000] ...
  [2900/10000] ...
  [3000/10000] ...
  [3100/10000] ...
  [3200/10000] ...
  [3300/10000] ...
  [3400/10000] ...
  [3500/10000] ...
  [3600/10000] ...
  [3700/10000] ...
  [3800/10000] ...
  [3900/10000] ...
  [4000/10000] ...
  [4100/10000] ...
  [4200/10000] ...
  [4300/10000] ...
  [4400/10000] ...
  [4500/10000] ...
  [4

### Step 3: Scipy Polishing

In [8]:
# Scipy L-BFGS-B polishing around refined best
def objective(params):
    tp, ts, eps, d = params
    res = evaluate_thresh(records, tp, ts, eps, d)
    return -res[metric]

tp_lo = max(tau_prior_min, best['tau_prior'] - tp_step / 2)
tp_hi = min(tau_prior_max, best['tau_prior'] + tp_step / 2)
ts_lo = max(tau_softmax_min, best['tau_softmax'] - ts_step / 2)
ts_hi = min(tau_softmax_max, best['tau_softmax'] + ts_step / 2)
eps_lo = max(eps_min, best['epsilon'] - eps_step_size / 2)
eps_hi = min(eps_max, best['epsilon'] + eps_step_size / 2)
d_lo_b = max(0.0, best['delta'] - (fine_delta[1] - fine_delta[0]))
d_hi_b = best['delta'] + (fine_delta[1] - fine_delta[0])

result = minimize(objective,
                  x0=[best['tau_prior'], best['tau_softmax'], best['epsilon'], best['delta']],
                  method='L-BFGS-B',
                  bounds=[(tp_lo, tp_hi), (ts_lo, ts_hi), (eps_lo, eps_hi), (d_lo_b, d_hi_b)])

tp_opt, ts_opt, eps_opt, d_opt = result.x
final = evaluate_thresh(records, tp_opt, ts_opt, eps_opt, d_opt)
print(f'Scipy best: tau_prior={tp_opt:.4f} tau_softmax={ts_opt:.4f} eps={eps_opt:.6f} delta={d_opt:.4f}')
print(f'  combo_r={final["combo_r"]:.4f}  marg_r={final["marg_r"]:.4f}  mean_ll={final["mean_ll"]:.4f}')

# Use scipy result if it improved, otherwise keep grid best
if final[metric] > best[metric]:
    best = final
    print('  -> Scipy improved over grid')
else:
    print('  -> Grid result retained')

Scipy best: tau_prior=6.0025 tau_softmax=4.6337 eps=0.933333 delta=34.9495
  combo_r=0.4970  marg_r=0.6042  mean_ll=-22.1345
  -> Scipy improved over grid


## 5. Switch-Stage Fine-Tuning

Fine-tune to maximize correlation specifically on switch stages (where at least one player changed role).
Model still conditions on full history — only the optimization target changes.

In [9]:
def filter_switch_stages(all_results):
    """Keep only predictions where the human combo changed from the previous stage."""
    filtered = {}
    for env_id, data in all_results.items():
        new_team_preds = []
        for team_preds in data['team_predictions']:
            filtered_preds = []
            for s, pred in enumerate(team_preds):
                if s > 0 and pred['human_combo'] != team_preds[s - 1]['human_combo']:
                    filtered_preds.append(pred)
            new_team_preds.append(filtered_preds)

        canon_combos = data['canonical_combos']
        stat_profile = data['stat_profile']
        stage_predicted = defaultdict(lambda: defaultdict(float))
        stage_human = defaultdict(lambda: defaultdict(int))
        stage_model_marg = defaultdict(lambda: np.zeros(3))
        stage_human_marg = defaultdict(lambda: np.zeros(3))
        stage_counts = defaultdict(int)
        max_stages = 0

        for team_preds in new_team_preds:
            for s, pred in enumerate(team_preds):
                stage_counts[s] += 1
                max_stages = max(max_stages, s + 1)
                for combo, prob in pred['predicted_dist'].items():
                    stage_predicted[s][canonical_combo(combo, stat_profile)] += prob
                stage_human[s][canonical_combo(pred['human_combo'], stat_profile)] += 1
                stage_model_marg[s] += pred['model_marginal']
                stage_human_marg[s] += combo_marginal(pred['human_combo'])

        if max_stages == 0:
            continue

        predicted_avg, model_marg_avg, human_marg_avg = {}, {}, {}
        for s in range(max_stages):
            n = stage_counts[s]
            if n > 0:
                predicted_avg[s] = {cc: stage_predicted[s].get(cc, 0.0) / n for cc in canon_combos}
                model_marg_avg[s] = stage_model_marg[s] / n
                human_marg_avg[s] = stage_human_marg[s] / n

        filtered[env_id] = dict(data)
        filtered[env_id].update({
            'max_stages': max_stages,
            'stage_predicted': predicted_avg,
            'stage_human': dict(stage_human),
            'stage_counts': dict(stage_counts),
            'team_predictions': new_team_preds,
            'stage_model_marginal': model_marg_avg,
            'stage_human_marginal': human_marg_avg,
        })

    return filtered


def evaluate_thresh_switch(records, tau_prior, tau_softmax, epsilon, delta):
    """Run on full history, compute metrics only on switch stages."""
    full_results = run_predictions(records, make_bayesian_thresh(
        tau_prior=tau_prior, tau_softmax=tau_softmax,
        epsilon=epsilon, delta=delta))
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'tau_prior': tau_prior, 'tau_softmax': tau_softmax,
                'epsilon': epsilon, 'delta': delta,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_prior': tau_prior, 'tau_softmax': tau_softmax,
        'epsilon': epsilon, 'delta': delta,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def evaluate_switch_from_cache_thresh(records, trajectories, tau_softmax, delta):
    """Evaluate switch-stage metrics using precomputed trajectories."""
    def predict_fn(record):
        idx = record['_traj_idx']
        return predict_from_trajectory_thresh(record, trajectories[idx],
                                              record['env_config']['values'],
                                              tau_softmax, delta)
    full_results = run_predictions(records, predict_fn)
    sw_results = filter_switch_stages(full_results)
    if not sw_results:
        return {'tau_softmax': tau_softmax, 'delta': delta,
                'combo_r': float('nan'), 'marg_r': float('nan'), 'mean_ll': float('nan')}
    corrs = compute_pearson(sw_results)
    ll = compute_log_likelihood(sw_results)
    g = corrs.get('__global__', {})
    return {
        'tau_softmax': tau_softmax, 'delta': delta,
        'combo_r': g.get('combo', {}).get('r', float('nan')),
        'marg_r': g.get('marginal', {}).get('r', float('nan')),
        'mean_ll': float(np.mean([v['mean_ll'] for v in ll.values()])) if ll else float('nan'),
    }


def grid_search_thresh_switch(records, tau_prior_vals, tau_softmax_vals, eps_vals, delta_vals):
    """Cached grid search optimizing switch-stage correlation."""
    for i, rec in enumerate(records):
        rec['_traj_idx'] = i

    results = []
    outer_combos = list(product(tau_prior_vals, eps_vals))
    inner_combos = list(product(tau_softmax_vals, delta_vals))
    total = len(outer_combos) * len(inner_combos)
    count = 0

    for tp, eps in outer_combos:
        trajectories = precompute_trajectories(records, tp, eps)
        for ts, d in inner_combos:
            count += 1
            if count % 100 == 0 or count == total:
                print(f'  [{count}/{total}] ...', flush=True)
            res = evaluate_switch_from_cache_thresh(records, trajectories, ts, d)
            res['tau_prior'] = tp
            res['epsilon'] = eps
            results.append(res)
    return results

In [10]:
# Coarse grid for switch-stage tuning (same ranges as aggregate)
print('=== Switch-Stage Fine-Tuning: Coarse Grid ===')
sw_total = len(tau_prior_vals) * len(tau_softmax_vals) * len(eps_vals) * len(delta_vals)
print(f'{sw_total} points\n')

sw_sweep = grid_search_thresh_switch(records, tau_prior_vals, tau_softmax_vals, eps_vals, delta_vals)
best_sw = pick_best(sw_sweep, metric)
print(f'\nSwitch coarse best: tau_prior={best_sw["tau_prior"]:.4f} tau_softmax={best_sw["tau_softmax"]:.4f} '
      f'eps={best_sw["epsilon"]:.6f} delta={best_sw["delta"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

=== Switch-Stage Fine-Tuning: Coarse Grid ===
23100 points

  [100/23100] ...
  [200/23100] ...
  [300/23100] ...
  [400/23100] ...
  [500/23100] ...
  [600/23100] ...
  [700/23100] ...
  [800/23100] ...
  [900/23100] ...
  [1000/23100] ...
  [1100/23100] ...
  [1200/23100] ...
  [1300/23100] ...
  [1400/23100] ...
  [1500/23100] ...
  [1600/23100] ...
  [1700/23100] ...
  [1800/23100] ...
  [1900/23100] ...
  [2000/23100] ...
  [2100/23100] ...
  [2200/23100] ...
  [2300/23100] ...
  [2400/23100] ...
  [2500/23100] ...
  [2600/23100] ...
  [2700/23100] ...
  [2800/23100] ...
  [2900/23100] ...
  [3000/23100] ...
  [3100/23100] ...
  [3200/23100] ...
  [3300/23100] ...
  [3400/23100] ...
  [3500/23100] ...
  [3600/23100] ...
  [3700/23100] ...
  [3800/23100] ...
  [3900/23100] ...
  [4000/23100] ...
  [4100/23100] ...
  [4200/23100] ...
  [4300/23100] ...
  [4400/23100] ...
  [4500/23100] ...
  [4600/23100] ...
  [4700/23100] ...
  [4800/23100] ...
  [4900/23100] ...
  [5000/23100] ...

In [11]:
# Refined grid for switch-stage tuning
fine_sw_tp = np.linspace(
    max(tau_prior_min, best_sw['tau_prior'] - tp_step),
    min(tau_prior_max, best_sw['tau_prior'] + tp_step), 10)
fine_sw_ts = np.linspace(
    max(tau_softmax_min, best_sw['tau_softmax'] - ts_step),
    min(tau_softmax_max, best_sw['tau_softmax'] + ts_step), 10)
fine_sw_eps = np.linspace(
    max(eps_min, best_sw['epsilon'] - eps_step_size),
    min(eps_max, best_sw['epsilon'] + eps_step_size), 10)
fine_sw_delta = np.linspace(
    max(delta_min, best_sw['delta'] - delta_step),
    min(delta_max, best_sw['delta'] + delta_step), 10)

print(f'Refined grid: {len(fine_sw_tp) * len(fine_sw_ts) * len(fine_sw_eps) * len(fine_sw_delta)} points')

sw_sweep2 = grid_search_thresh_switch(records, fine_sw_tp, fine_sw_ts, fine_sw_eps, fine_sw_delta)
best_sw = pick_best(sw_sweep2, metric)
print(f'\nSwitch refined best: tau_prior={best_sw["tau_prior"]:.4f} tau_softmax={best_sw["tau_softmax"]:.4f} '
      f'eps={best_sw["epsilon"]:.6f} delta={best_sw["delta"]:.4f}')
print(f'  combo_r={best_sw["combo_r"]:.4f}  marg_r={best_sw["marg_r"]:.4f}  mean_ll={best_sw["mean_ll"]:.4f}')

Refined grid: 10000 points
  [100/10000] ...
  [200/10000] ...
  [300/10000] ...
  [400/10000] ...
  [500/10000] ...
  [600/10000] ...
  [700/10000] ...
  [800/10000] ...
  [900/10000] ...
  [1000/10000] ...
  [1100/10000] ...
  [1200/10000] ...
  [1300/10000] ...
  [1400/10000] ...
  [1500/10000] ...
  [1600/10000] ...
  [1700/10000] ...
  [1800/10000] ...
  [1900/10000] ...
  [2000/10000] ...
  [2100/10000] ...
  [2200/10000] ...
  [2300/10000] ...
  [2400/10000] ...
  [2500/10000] ...
  [2600/10000] ...
  [2700/10000] ...
  [2800/10000] ...
  [2900/10000] ...
  [3000/10000] ...
  [3100/10000] ...
  [3200/10000] ...
  [3300/10000] ...
  [3400/10000] ...
  [3500/10000] ...
  [3600/10000] ...
  [3700/10000] ...
  [3800/10000] ...
  [3900/10000] ...
  [4000/10000] ...
  [4100/10000] ...
  [4200/10000] ...
  [4300/10000] ...
  [4400/10000] ...
  [4500/10000] ...
  [4600/10000] ...
  [4700/10000] ...
  [4800/10000] ...
  [4900/10000] ...
  [5000/10000] ...
  [5100/10000] ...
  [5200/10000

In [12]:
# Scipy polishing for switch-stage tuning
def objective_sw(params):
    tp, ts, eps, d = params
    res = evaluate_thresh_switch(records, tp, ts, eps, d)
    return -res[metric]

sw_tp_lo = max(tau_prior_min, best_sw['tau_prior'] - tp_step / 2)
sw_tp_hi = min(tau_prior_max, best_sw['tau_prior'] + tp_step / 2)
sw_ts_lo = max(tau_softmax_min, best_sw['tau_softmax'] - ts_step / 2)
sw_ts_hi = min(tau_softmax_max, best_sw['tau_softmax'] + ts_step / 2)
sw_eps_lo = max(eps_min, best_sw['epsilon'] - eps_step_size / 2)
sw_eps_hi = min(eps_max, best_sw['epsilon'] + eps_step_size / 2)
sw_d_lo = max(0.0, best_sw['delta'] - (fine_sw_delta[1] - fine_sw_delta[0]))
sw_d_hi = best_sw['delta'] + (fine_sw_delta[1] - fine_sw_delta[0])

result_sw = minimize(objective_sw,
                     x0=[best_sw['tau_prior'], best_sw['tau_softmax'],
                         best_sw['epsilon'], best_sw['delta']],
                     method='L-BFGS-B',
                     bounds=[(sw_tp_lo, sw_tp_hi), (sw_ts_lo, sw_ts_hi),
                             (sw_eps_lo, sw_eps_hi), (sw_d_lo, sw_d_hi)])

tp_sw, ts_sw, eps_sw, d_sw = result_sw.x
final_sw = evaluate_thresh_switch(records, tp_sw, ts_sw, eps_sw, d_sw)
print(f'Scipy best: tau_prior={tp_sw:.4f} tau_softmax={ts_sw:.4f} eps={eps_sw:.6f} delta={d_sw:.4f}')
print(f'  combo_r={final_sw["combo_r"]:.4f}  marg_r={final_sw["marg_r"]:.4f}  mean_ll={final_sw["mean_ll"]:.4f}')

if final_sw[metric] > best_sw[metric]:
    best_sw = final_sw
    print('  -> Scipy improved over grid')
else:
    print('  -> Grid result retained')

Scipy best: tau_prior=20.0000 tau_softmax=0.1000 eps=0.075003 delta=0.0000
  combo_r=0.2493  marg_r=0.3903  mean_ll=-39.9226
  -> Scipy improved over grid


## 6. Save Parameters

In [13]:
output = {
    'metric_optimized': metric,
    'aggregate_tuned': {
        'bayesian_thresh': {
            'tau_prior': best['tau_prior'], 'tau_softmax': best['tau_softmax'],
            'epsilon': best['epsilon'], 'delta': best['delta'],
            'combo_r': best['combo_r'], 'marg_r': best['marg_r'], 'mean_ll': best['mean_ll'],
        },
    },
    'switch_stage_tuned': {
        'bayesian_thresh': {
            'tau_prior': best_sw['tau_prior'], 'tau_softmax': best_sw['tau_softmax'],
            'epsilon': best_sw['epsilon'], 'delta': best_sw['delta'],
            'combo_r': best_sw['combo_r'], 'marg_r': best_sw['marg_r'], 'mean_ll': best_sw['mean_ll'],
        },
    },
}

with open('bayesian_thresh_params.json', 'w') as f:
    json.dump(output, f, indent=2)
print('Saved to bayesian_thresh_params.json')
print(json.dumps(output, indent=2))

Saved to bayesian_thresh_params.json
{
  "metric_optimized": "combo_r",
  "aggregate_tuned": {
    "bayesian_thresh": {
      "tau_prior": 6.002517856891009,
      "tau_softmax": 4.633650360095246,
      "epsilon": 0.9333333333333333,
      "delta": 34.94949494949495,
      "combo_r": 0.49695093159148823,
      "marg_r": 0.6042082910744191,
      "mean_ll": -22.13448502643374
    }
  },
  "switch_stage_tuned": {
    "bayesian_thresh": {
      "tau_prior": 20.0,
      "tau_softmax": 0.1,
      "epsilon": 0.07500294620955639,
      "delta": 0.0,
      "combo_r": 0.2492592088209796,
      "marg_r": 0.3902972567333126,
      "mean_ll": -39.92258731145132
    }
  }
}
